In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dotenv import load_dotenv

# load_dotenv()

# Docking Workflow

This notebook demonstrates how to perform molecular docking using Deep Origin's drug discovery platform. You'll learn how to:

1. **Load and prepare proteins** - Load a protein structure and prepare it for docking
2. **Find binding pockets** - Identify potential binding sites on the protein
3. **Dock ligands** - Perform docking calculations for single or multiple ligands
4. **Monitor jobs** - Track the progress of docking calculations
5. **Analyze results** - Visualize and filter docking poses

Let's get started!


## Setup

First, we'll import the necessary Deep Origin drug discovery modules.


In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Pocket,
    Protein,
    Docking,
    Ligand,
LigandSet,
)
from deeporigin.platform import DeepOriginClient
import deeporigin

deeporigin.__version__

In [ ]:
client = DeepOriginClient()
client

## Load Protein Structure

Here we load a protein structure from a PDB file. The `Complex` object represents a protein-ligand complex and will be used throughout the docking workflow. 




In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.remove_water()
protein.sync()
protein.id

In [ ]:
protein.name

## Load Ligands

Load a set of ligands from a CSV file containing SMILES strings. The `LigandSet` object allows you to work with multiple ligands at once. You can visualize them in a grid to see what molecules you're working with.


In [ ]:
ligands = LigandSet.from_dir(BRD_DATA_DIR)
ligands.sync()
ligands

In [ ]:
ligands.to_smiles()

In [ ]:
ligands.show_grid()

## Protonate Ligands

It is reccomended that ligands are protonated before running docking

## Assign Ligands to Complex

Associate the ligands with the protein complex. This prepares the system for docking calculations.


## Visualize the Protein

Display the protein structure in 3D. This helps you understand the protein's structure before proceeding with docking.


## Find Pockets

Use ``PocketFinder`` from ``deeporigin.drug_discovery`` to detect cavities and potential binding sites on the protein surface.

In [ ]:
pockets = Pocket.from_result(protein_id=protein.id)
pocket = pockets[0]
pocket

## Inspect Binding Pockets

View the detected binding pockets. Each pocket represents a potential binding site. You'll typically want to dock ligands into the most promising pocket (often the largest or most druggable one).


## Bulk Docking Workflow

For drug discovery, you'll often want to dock many ligands at once. The bulk docking workflow allows you to:

1. **Submit multiple docking jobs** - Dock all ligands in your ligand set
2. **Monitor progress** - Track job status in real-time
3. **Retrieve results** - Download all poses once calculations complete
4. **Analyze at scale** - Compare binding across all ligands

The `run()` method with `quote=True` first provides a cost estimate before submitting jobs. You can specify:
- **pocket**: Which binding pocket to use
- **batch_size**: How many ligands to process per batch



In [ ]:
docking = Docking(protein=protein, pocket=pocket, ligands=ligands)
docking

In [ ]:
docking.start()

In [ ]:
docking.name

In [ ]:
await docking.watch()

In [ ]:
docking.sync()
docking.progress

In [ ]:
ligands = LigandSet.from_sdf("/Users/srinivas/code/cli/tests/fixtures/42-ligands.sdf")
ligands.sync()

In [ ]:
docking = Docking(ligands=ligands, protein=protein, pocket=pocket)
docking.start()


In [ ]:
task = await docking.watch()

In [ ]:
docking.sync()
docking.progress

## Review Job Details

Before confirming, review the job details including:
- Number of ligands to dock
- Estimated cost
- Expected completion time



Use `confirm()` to submit the jobs for execution.


In [ ]:
docking.quote()
docking.estimate

## Monitor Job Progress

The `watch()` method monitors your docking jobs and updates you on their progress. It will:
- Check job status at regular intervals
- Display progress updates
- Notify you when jobs complete

You can cancel jobs if needed using `jobs.cancel()`.


In [ ]:
docking.start()

In [ ]:
await docking.watch()

In [ ]:
len(client.results.get(compute_job_id="10051f85-04ff-41d5-88ad-0c8e1d7a385a", limit=1000)["data"])

In [ ]:
docking.sync()
docking.progress

In [ ]:
client.progress_reports.get(execution_id="local840-fe7d-4237-813f-9469c9d4cdeb")["data"]

## Retrieve Docking Results

Once jobs complete, retrieve all poses using `get_poses()`. This downloads all calculated poses for all ligands in your set.


In [ ]:
docking = Docking.from_id("ce413cfb-da2c-491b-9bf5-f5935e5d08c7")

In [ ]:
docking.name

## Convert to DataFrame for Analysis

Convert poses to a DataFrame for detailed analysis. This enables:
- Statistical analysis of binding energies
- Comparison across ligands
- Filtering and sorting
- Export to CSV or other formats


In [ ]:
df = poses.to_dataframe()
df

## Visualize Statistics of All Poses

Create a scatter plot showing all poses from all docked ligands. The plot displays binding energy vs Pose Score. Hover over each point to see details about the ligand and pose.


In [ ]:
poses.plot()

## Visualize Statistics of Best Poses

Display the top pose for each ligand in the protein structure. This gives you a visual overview of how different ligands bind to the protein, helping you identify promising candidates for further study.



In [ ]:
top_poses = poses.filter_top_poses()
top_poses.plot()

## Show best poses

Find the best pose for each ligand and visualize their conformations in the protein structure. This helps identify the most promising binding modes across your ligand set.

In [ ]:
sim.protein.show(poses=top_poses)